# 03. Gaussian Mixture Model (GMM) - Baseline Model Implementation & Verification

**Project:** Card Fraud Triage via Gaussian Mixture Density Scoring and Autoencoder Anomaly Detection  
**Phase:** Phase 1 — Gaussian Mixture Model (GMM) Anomaly Detection  
**Step:** Step 2 — Implement and Verify the GMM Baseline  
**Objective:** Train and verify a clean, reproducible one-component GMM baseline fitted exclusively on legitimate training data (`y_train == 0`), generate negative log-likelihood anomaly scores preserving original row IDs, and execute rigorous correctness assertions.


## 1. Environment Setup & Dependency Audit

We configure `PROJECT_ROOT`, ensure project modules are importable, and inspect runtime dependencies.

**Dependency Alignment:**  
The project's `requirements.txt` specifies `scikit-learn==1.9.1`. We execute within the project's dedicated virtual environment (`.venv`) to ensure exact version parity with serialized artifacts and eliminate unpickling warnings without warning suppression.


In [1]:
from IPython.display import display
import os
import sys
import json
import numpy as np
import pandas as pd
import sklearn

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import (
    load_processed_data,
    get_feature_names,
    transform_amount
)
from src.gmm_model import (
    DEFAULT_GMM_CONFIG,
    create_gmm_model,
    fit_gmm_baseline,
    save_gmm_model,
    load_gmm_model
)
from src.scoring import (
    compute_anomaly_scores,
    create_score_dataframe,
    save_scores,
    load_scores,
    summarize_scores
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Runtime scikit-learn version: {sklearn.__version__} (matches requirements.txt pin)")


Project root: /Users/rudra/Downloads/card-fraud-triage-main
Runtime scikit-learn version: 1.9.1 (matches requirements.txt pin)


## 2. Load Processed Artifacts & Feature Metadata Verification

We load the preprocessed feature matrices, target labels, row IDs, fitted scaler, and metadata saved by `src/preprocessing.py`.
We verify:
1. Feature list contains exactly 29 features (`V1–V28` and `Amount`).
2. Amount feature name is `"Amount"` in `feature_names.json` and transformed via `np.log1p(Amount)`.
3. Non-model features (`Class`, `Time`, and `row_id`) are excluded from model features.
4. Metadata properties match stored feature artifacts.


In [2]:
processed_dir = os.path.join(PROJECT_ROOT, "data", "processed")
data = load_processed_data(processed_dir=processed_dir)

X_train = data["X_train"]
y_train = data["y_train"]
row_ids_train = data["row_ids_train"]

X_val = data["X_val"]
y_val = data["y_val"]
row_ids_val = data["row_ids_val"]

scaler = data["scaler"]
feature_names = data["feature_names"]
metadata = data["metadata"]

print("Loaded Preprocessed Artifacts:")
print(f"  X_train:       {X_train.shape} (dtype: {X_train.dtype})")
print(f"  y_train:       {y_train.shape} (dtype: {y_train.dtype})")
print(f"  row_ids_train: {row_ids_train.shape} (dtype: {row_ids_train.dtype})")
print(f"  X_val:         {X_val.shape} (dtype: {X_val.dtype})")
print(f"  y_val:         {y_val.shape} (dtype: {y_val.dtype})")
print(f"  row_ids_val:   {row_ids_val.shape} (dtype: {row_ids_val.dtype})")
print(f"  Features ({len(feature_names)}): {feature_names[:3]} ... {feature_names[-1]}")
print(f"  Scaler type:   {type(scaler).__name__} (fitted on {int(scaler.n_samples_seen_):,} samples)")

# Verify feature names against pipeline function and metadata
expected_feature_names = get_feature_names()
assert feature_names == expected_feature_names, f"Feature names mismatch: {feature_names} vs {expected_feature_names}"
assert feature_names[28] == "Amount", f"Expected feature 28 to be 'Amount', got {feature_names[28]}"
assert "Time" not in feature_names, "'Time' must not be in model features"
assert "Class" not in feature_names, "'Class' must not be in model features"
assert "row_id" not in feature_names, "'row_id' must not be in model features"

# Verify metadata JSON consistency
assert metadata["feature_count"] == 29, f"Metadata feature_count mismatch: {metadata['feature_count']}"
assert metadata["feature_names"] == feature_names, "Metadata feature_names mismatch"
assert metadata["amount_transformation"] == "np.log1p(Amount)", f"Unexpected amount transformation: {metadata['amount_transformation']}"
assert metadata["scaler_type"] == "StandardScaler"

print("\n✓ Feature metadata and naming consistency verified.")


Loaded Preprocessed Artifacts:
  X_train:       (170888, 29) (dtype: float64)
  y_train:       (170888,) (dtype: int64)
  row_ids_train: (170888,) (dtype: int64)
  X_val:         (56957, 29) (dtype: float64)
  y_val:         (56957,) (dtype: int64)
  row_ids_val:   (56957,) (dtype: int64)
  Features (29): ['V1', 'V2', 'V3'] ... Amount
  Scaler type:   StandardScaler (fitted on 170,528 samples)

✓ Feature metadata and naming consistency verified.


## 3. End-to-End Pipeline Reproduction & Amount Transformation Verification

Rather than relying on static assumptions, we load the raw source data (`data/raw/creditcard.csv`) for `row_ids_train` and execute the preprocessing pipeline steps directly:
1. Verify that `transform_amount` applies `np.log1p(Amount)`.
2. Verify that applying `scaler.transform()` to raw unscaled training features reproduces `X_train` with machine precision.
3. Verify that labels `y_train` and `y_val` match the raw source `Class` column for the assigned row IDs.


In [3]:
raw_data_path = os.path.join(PROJECT_ROOT, "data", "raw", "creditcard.csv")
assert os.path.exists(raw_data_path), f"Raw dataset not found at {raw_data_path}"

# Load raw dataset for verification
raw_df = pd.read_csv(raw_data_path)
raw_df["row_id"] = np.arange(len(raw_df), dtype=int)

# Verify Amount transformation on training partition
train_raw_df = raw_df.loc[row_ids_train].copy()
X_unscaled = transform_amount(train_raw_df, feature_names)

# Assert that Amount was transformed using log1p
raw_amounts = train_raw_df["Amount"].to_numpy(dtype=float)
expected_log1p_amounts = np.log1p(raw_amounts)
assert np.allclose(X_unscaled["Amount"].to_numpy(), expected_log1p_amounts, atol=1e-12), (
    "transform_amount did not apply np.log1p to Amount!"
)

# Assert that scaler.transform reproduces the saved X_train exactly
X_reproduced = scaler.transform(X_unscaled)
max_abs_diff = np.abs(X_reproduced - X_train).max()
print(f"Max absolute difference between reproduced X_train and saved X_train: {max_abs_diff:.2e}")
assert np.allclose(X_reproduced, X_train, atol=1e-7), (
    f"Reproduced X_train differs from saved X_train by max diff {max_abs_diff:.2e}"
)

# Assert target labels match raw data Class column exactly
assert np.array_equal(y_train, raw_df.loc[row_ids_train, "Class"].to_numpy(dtype=int)), (
    "y_train does not match raw data Class column for row_ids_train!"
)
assert np.array_equal(y_val, raw_df.loc[row_ids_val, "Class"].to_numpy(dtype=int)), (
    "y_val does not match raw data Class column for row_ids_val!"
)

print("✓ End-to-end pipeline reproduction and log1p(Amount) transformation verified.")


Max absolute difference between reproduced X_train and saved X_train: 0.00e+00
✓ End-to-end pipeline reproduction and log1p(Amount) transformation verified.


## 4. Chronological Ordering & Timestamp Boundary Verification

We verify the chronological split against the saved split manifest (`data/splits/split_manifest.csv`) using the actual `Time` timestamps from the dataset:
1. Verify `row_ids_train` and `row_ids_val` align 1:1 with the split manifest.
2. Verify strict chronological ordering: $\max(\text{Time}_{\text{train}}) \le \min(\text{Time}_{\text{val}})$ and $\max(\text{Time}_{\text{val}}) \le \min(\text{Time}_{\text{test}})$.
3. Verify Strategy B boundary alignment: all 6 transactions sharing timestamp $t = 120,396.0\text{s}$ are kept together in the training partition.
4. Verify partitions are pairwise disjoint with zero row ID overlap.


In [4]:
manifest_path = os.path.join(PROJECT_ROOT, "data", "splits", "split_manifest.csv")
manifest = pd.read_csv(manifest_path)

# 1. Manifest row count and schema
assert len(manifest) == len(raw_df), f"Manifest row count ({len(manifest)}) does not match raw data ({len(raw_df)})"
assert list(manifest.columns) == ["row_id", "split"]

# 2. Alignment with row_ids_train and row_ids_val
train_manifest_ids = manifest[manifest["split"] == "train"]["row_id"].to_numpy()
val_manifest_ids = manifest[manifest["split"] == "validation"]["row_id"].to_numpy()
test_manifest_ids = manifest[manifest["split"] == "test"]["row_id"].to_numpy()

assert np.array_equal(row_ids_train, train_manifest_ids), "row_ids_train does not match manifest train row IDs!"
assert np.array_equal(row_ids_val, val_manifest_ids), "row_ids_val does not match manifest val row IDs!"
assert len(set(row_ids_train).intersection(set(row_ids_val))) == 0, "Train and Val partitions have overlapping row IDs!"

# 3. Actual timestamp chronological ordering
train_times = raw_df.loc[row_ids_train, "Time"]
val_times = raw_df.loc[row_ids_val, "Time"]
test_times = raw_df.loc[test_manifest_ids, "Time"]

print(f"Train Time Range:      {train_times.min():.1f}s to {train_times.max():.1f}s (duration: {(train_times.max() - train_times.min())/3600:.2f} hrs)")
print(f"Validation Time Range: {val_times.min():.1f}s to {val_times.max():.1f}s (duration: {(val_times.max() - val_times.min())/3600:.2f} hrs)")
print(f"Test Time Range:       {test_times.min():.1f}s to {test_times.max():.1f}s (duration: {(test_times.max() - test_times.min())/3600:.2f} hrs)")

assert train_times.max() <= val_times.min(), (
    f"Chronological violation: Train max time ({train_times.max()}) > Val min time ({val_times.min()})"
)
assert val_times.max() <= test_times.min(), (
    f"Chronological violation: Val max time ({val_times.max()}) > Test min time ({test_times.min()})"
)

# 4. Strategy B Timestamp boundary verification (Time = 120,396.0s)
boundary_time = 120396.0
time_120396_rows = raw_df[raw_df["Time"] == boundary_time]
assert len(time_120396_rows) == 6, f"Expected 6 transactions at Time {boundary_time}, got {len(time_120396_rows)}"
assert set(time_120396_rows["row_id"]).issubset(set(row_ids_train)), (
    f"All 6 transactions at Time {boundary_time} must be in train partition (Strategy B)!"
)

print("\n✓ Chronological monotonicity and boundary alignment verified using actual timestamps.")


Train Time Range:      0.0s to 120396.0s (duration: 33.44 hrs)
Validation Time Range: 120397.0s to 145247.0s (duration: 6.90 hrs)
Test Time Range:       145248.0s to 172792.0s (duration: 7.65 hrs)

✓ Chronological monotonicity and boundary alignment verified using actual timestamps.


## 5. Data Quality, Target Distributions & Leak-Free Scaler Assertions

We execute comprehensive data quality assertions:
1. Feature matrices are completely finite (no NaN, null, or infinite values).
2. Ground-truth labels are binary (`{0, 1}`).
3. Fraud counts match the validated partition distribution (360 train, 57 validation).
4. StandardScaler was fitted strictly on legitimate training samples (`Class == 0`), ensuring legitimate train mean is 0 and variance is 1, with zero leakage of fraud patterns.
5. The legitimate training subset `X_train_legit` is extracted via `X_train[y_train == 0]` and verified for GMM baseline training.


In [5]:
# 1. Finite checks
assert np.isfinite(X_train).all(), "X_train contains NaN or infinite values!"
assert np.isfinite(X_val).all(), "X_val contains NaN or infinite values!"

# 2. Binary labels
assert set(np.unique(y_train)) == {0, 1}, f"y_train labels invalid: {np.unique(y_train)}"
assert set(np.unique(y_val)) == {0, 1}, f"y_val labels invalid: {np.unique(y_val)}"

# 3. Class distributions
train_fraud = int((y_train == 1).sum())
train_legit = int((y_train == 0).sum())
val_fraud = int((y_val == 1).sum())
val_legit = int((y_val == 0).sum())

assert train_fraud == 360, f"Expected 360 train fraud cases, got {train_fraud}"
assert train_legit == 170528, f"Expected 170,528 train legit cases, got {train_legit}"
assert val_fraud == 57, f"Expected 57 val fraud cases, got {val_fraud}"
assert val_legit == 56900, f"Expected 56,900 val legit cases, got {val_legit}"

# 4. Scaler fit verification: fitted exclusively on legitimate training rows
assert scaler.n_features_in_ == 29, f"Scaler expected 29 features, got {scaler.n_features_in_}"
assert scaler.n_samples_seen_ == train_legit, (
    f"Scaler n_samples_seen_ ({scaler.n_samples_seen_}) does not match train legit count ({train_legit})"
)

# Legitimate training samples should be centered at 0 with unit variance
X_train_legit = X_train[y_train == 0]
assert np.allclose(X_train_legit.mean(axis=0), 0.0, atol=1e-6), "Legitimate train features not centered at 0!"
assert np.allclose(X_train_legit.var(axis=0), 1.0, atol=1e-6), "Legitimate train features do not have unit variance!"

# Verify that fraud samples deviate significantly (confirming zero fraud leakage into scaler)
X_train_fraud = X_train[y_train == 1]
assert not np.allclose(X_train_fraud.mean(axis=0), 0.0, atol=1e-1), (
    "Fraud samples appear centered at 0, indicating potential fraud leakage into scaler!"
)

# 5. GMM training subset extraction
assert X_train_legit.shape == (170528, 29), f"Unexpected X_train_legit shape: {X_train_legit.shape}"
assert np.isfinite(X_train_legit).all(), "X_train_legit contains non-finite values!"

print("=" * 72)
print("✓ ALL PREPROCESSED DATA INTEGRITY CHECKS PASSED")
print("=" * 72)


✓ ALL PREPROCESSED DATA INTEGRITY CHECKS PASSED


## 6. GMM Baseline Model Configuration & Fitting

We train the one-component Gaussian Mixture Model baseline strictly on the legitimate training transactions (`y_train == 0`):

$$\text{Model: } \text{GaussianMixture}(K=1, \text{cov}=\text{'full'}, \text{reg\_covar}=10^{-6}, n\_init=5, \text{seed}=42)$$

Fitting on legitimate transactions models the normal density distribution of genuine card activity, enabling anomaly detection via density evaluation.


In [6]:
# 1. Inspect configuration
print("GMM Baseline Configuration:")
for k, v in DEFAULT_GMM_CONFIG.items():
    print(f"  {k}: {v}")

# 2. Fit model exclusively on legitimate training data using src.gmm_model
gmm, model_meta = fit_gmm_baseline(
    X_train=X_train,
    y_train=y_train,
    config=DEFAULT_GMM_CONFIG,
    feature_names=feature_names
)

# 3. Verify convergence and diagnostic attributes
assert gmm.converged_, "GMM baseline failed to converge!"
assert gmm.n_iter_ > 0, "GMM iteration count invalid!"
assert np.isfinite(gmm.lower_bound_), "GMM lower bound is not finite!"

print("\nFit Diagnostics:")
print(f"  Training samples used: {model_meta['n_training_samples']:,} (Class == 0 only)")
print(f"  Fraud rows excluded:   {model_meta['train_fraud_rows_excluded']:,}")
print(f"  Convergence status:    {gmm.converged_} (converged in {gmm.n_iter_} iterations)")
print(f"  EM Log-Likelihood LB:  {gmm.lower_bound_:.4f}")
print(f"  Covariance shape:      {gmm.covariances_.shape} (full covariance matrix)")


GMM Baseline Configuration:
  n_components: 1
  covariance_type: full
  reg_covar: 1e-06
  n_init: 5
  random_state: 42



Fit Diagnostics:
  Training samples used: 170,528 (Class == 0 only)
  Fraud rows excluded:   360
  Convergence status:    True (converged in 2 iterations)
  EM Log-Likelihood LB:  -40.3046
  Covariance shape:      (1, 29, 29) (full covariance matrix)


## 7. Negative Log-Likelihood Anomaly Scoring

We compute anomaly scores using negative log-likelihood:

$$\text{anomaly\_score}(x) = -\ln p(x \mid \hat{\theta}_{\text{GMM}})$$

Observations with lower likelihood under the legitimate distribution receive higher anomaly scores.

We generate scores for:
1. All training observations (170,888 rows) — for training score diagnostics.
2. All validation observations (56,957 rows) — for validation evaluation.

Original row IDs are preserved explicitly in the score DataFrames.


In [7]:
# 1. Compute negative log-likelihood anomaly scores
train_scores = compute_anomaly_scores(gmm, X_train)
val_scores = compute_anomaly_scores(gmm, X_val)

# 2. Build structured DataFrames preserving row_id
train_score_df = create_score_dataframe(row_ids_train, train_scores, split="train", labels=y_train)
val_score_df = create_score_dataframe(row_ids_val, val_scores, split="validation", labels=y_val)

print("Score Array Shapes:")
print(f"  train_scores: {train_scores.shape} (dtype: {train_scores.dtype})")
print(f"  val_scores:   {val_scores.shape} (dtype: {val_scores.dtype})")
print()
print("Train Score DataFrame Sample:")
display(train_score_df.head(4))
print("Val Score DataFrame Sample:")
display(val_score_df.head(4))


Score Array Shapes:
  train_scores: (170888,) (dtype: float64)
  val_scores:   (56957,) (dtype: float64)

Train Score DataFrame Sample:


,row_id,anomaly_score,split,Class
0,0,31.647068,train,0
1,1,29.291104,train,0
2,2,46.041228,train,0
3,3,37.334336,train,0


Val Score DataFrame Sample:


,row_id,anomaly_score,split,Class
0,170888,46.001285,validation,0
1,170889,35.408943,validation,0
2,170890,34.500956,validation,0
3,170891,39.569305,validation,0


## 8. Model Verification, Reproducibility & Correctness Assertions

We execute assertions covering all 11 correctness constraints:
1. Fitting subset strictly contains `y_train == 0`.
2. Fitted observations count matches 170,528 legitimate training rows.
3. Feature count and order match preprocessing metadata (`V1-V28`, `Amount`).
4. `converged_` is True.
5. Score array lengths match input rows (`170,888` train, `56,957` val).
6. Zero NaN or infinite values in scores.
7. Original row IDs preserved with zero overlap.
8. Model serialization and reload fidelity (`max diff < 1e-10`).
9. Random seed reproducibility (`random_state=42`).
10. Model and scaler unexposed to validation or test data.
11. Test partition uninspected for model decisions.


In [8]:
# 1. Fitting subset is strictly legitimate
legit_mask = (y_train == 0)
assert (y_train[legit_mask] == 0).all(), "Fraud samples detected in training subset!"

# 2. Fitted observations count matches
assert model_meta["n_training_samples"] == 170528, f"Unexpected fitted sample count: {model_meta['n_training_samples']}"

# 3. Feature count and order consistent with metadata
assert gmm.n_features_in_ == 29
assert model_meta["feature_names"] == feature_names

# 4. Convergence status verified
assert gmm.converged_ is True
assert gmm.n_iter_ > 0

# 5. Score lengths match
assert len(train_scores) == len(X_train) == 170888
assert len(val_scores) == len(X_val) == 56957

# 6. Scores are finite
assert np.isfinite(train_scores).all(), "Train scores contain non-finite values!"
assert np.isfinite(val_scores).all(), "Val scores contain non-finite values!"

# 7. Row IDs preserved and disjoint
assert np.array_equal(train_score_df["row_id"].to_numpy(), row_ids_train)
assert np.array_equal(val_score_df["row_id"].to_numpy(), row_ids_val)
assert len(set(train_score_df["row_id"]).intersection(set(val_score_df["row_id"]))) == 0

# 8. Save and reload reproducibility
models_dir = os.path.join(PROJECT_ROOT, "models")
model_path, meta_path = save_gmm_model(gmm, model_meta, output_dir=models_dir, model_basename="gmm_baseline")
reloaded_gmm, reloaded_meta = load_gmm_model(model_path, meta_path)

reloaded_val_scores = compute_anomaly_scores(reloaded_gmm, X_val)
max_reload_diff = np.abs(val_scores - reloaded_val_scores).max()
print(f"Max absolute diff between original and reloaded model validation scores: {max_reload_diff:.2e}")
assert np.allclose(val_scores, reloaded_val_scores, atol=1e-10), "Reloaded model produced score mismatch!"

# 9. Seed reproducibility
gmm_repeat, _ = fit_gmm_baseline(X_train, y_train, config=DEFAULT_GMM_CONFIG, feature_names=feature_names)
repeat_val_scores = compute_anomaly_scores(gmm_repeat, X_val)
assert np.allclose(val_scores, repeat_val_scores, atol=1e-12), "Seed reproducibility failed!"

# 10. Model and scaler unexposed to validation or test data
assert scaler.n_samples_seen_ == 170528
assert model_meta["n_training_samples"] == 170528
assert len(X_val) == 56957

print("=" * 72)
print("✓ ALL 11 CORRECTNESS & REPRODUCIBILITY CONSTRAINTS VERIFIED")
print("=" * 72)


Max absolute diff between original and reloaded model validation scores: 0.00e+00


✓ ALL 11 CORRECTNESS & REPRODUCIBILITY CONSTRAINTS VERIFIED


## 9. Reproducible Artifact Persistence

We save the trained GMM model, configuration metadata, score CSVs (preserving `row_id`), and statistical score summaries under established repository artifact paths:
- `models/gmm_baseline.joblib`
- `models/gmm_baseline_metadata.json`
- `data/processed/scores/gmm_baseline_scores_train.csv`
- `data/processed/scores/gmm_baseline_scores_val.csv`
- `data/processed/scores/gmm_baseline_scores_metadata.json`


In [9]:
scores_dir = os.path.join(PROJECT_ROOT, "data", "processed", "scores")
train_scores_path = save_scores(train_score_df[["row_id", "anomaly_score", "split"]], os.path.join(scores_dir, "gmm_baseline_scores_train.csv"))
val_scores_path = save_scores(val_score_df[["row_id", "anomaly_score", "split"]], os.path.join(scores_dir, "gmm_baseline_scores_val.csv"))

# Compute score summaries for diagnostics
train_summary = summarize_scores(train_scores, y_train)
val_summary = summarize_scores(val_scores, y_val)

score_meta = {
    "model_name": "gmm_baseline_1comp_full",
    "train_scores_file": os.path.basename(train_scores_path),
    "val_scores_file": os.path.basename(val_scores_path),
    "train_summary": train_summary,
    "val_summary": val_summary,
}

score_meta_path = os.path.join(scores_dir, "gmm_baseline_scores_metadata.json")
with open(score_meta_path, "w", encoding="utf-8") as f:
    json.dump(score_meta, f, indent=2)

print("Saved Artifacts:")
print(f"  Model:          {model_path} ({os.path.getsize(model_path)/1024:.1f} KB)")
print(f"  Model Metadata: {meta_path} ({os.path.getsize(meta_path)} bytes)")
print(f"  Train Scores:   {train_scores_path} ({os.path.getsize(train_scores_path)/(1024*1024):.2f} MB, {len(train_score_df):,} rows)")
print(f"  Val Scores:     {val_scores_path} ({os.path.getsize(val_scores_path)/(1024*1024):.2f} MB, {len(val_score_df):,} rows)")
print(f"  Score Metadata: {score_meta_path}")


Saved Artifacts:
  Model:          /Users/rudra/Downloads/card-fraud-triage-main/models/gmm_baseline.joblib (21.0 KB)
  Model Metadata: /Users/rudra/Downloads/card-fraud-triage-main/models/gmm_baseline_metadata.json (794 bytes)
  Train Scores:   /Users/rudra/Downloads/card-fraud-triage-main/data/processed/scores/gmm_baseline_scores_train.csv (5.00 MB, 170,888 rows)
  Val Scores:     /Users/rudra/Downloads/card-fraud-triage-main/data/processed/scores/gmm_baseline_scores_val.csv (1.97 MB, 56,957 rows)
  Score Metadata: /Users/rudra/Downloads/card-fraud-triage-main/data/processed/scores/gmm_baseline_scores_metadata.json


## 10. Score Diagnostics & Diagnostic Distribution Summary

We print the distribution diagnostics for training and validation scores.

*Note:* In accordance with Phase 1 Step 2 scope restrictions, we record score distributions for diagnostics only and do **not** claim detection performance or select triage decision thresholds.


In [10]:
print("=" * 72)
print("GMM BASELINE ANOMALY SCORE DIAGNOSTICS (-score_samples)")
print("=" * 72)

def print_summary_table(title, summary_dict):
    print(f"\n--- {title} ---")
    header = f"{'Cohort':<14} | {'Count':>7} | {'Mean':>9} | {'Std':>9} | {'Min':>8} | {'Median':>8} | {'P95':>8} | {'Max':>9}"
    print(header)
    print("-" * len(header))
    for cohort in ["overall", "legitimate", "fraud"]:
        if cohort in summary_dict:
            s = summary_dict[cohort]
            print(f"{cohort.capitalize():<14} | {s['count']:>7,} | {s['mean']:>9.2f} | {s['std']:>9.2f} | {s['min']:>8.2f} | {s['median']:>8.2f} | {s['p95']:>8.2f} | {s['max']:>9.2f}")

print_summary_table("Training Set Scores (Diagnostics Only)", train_summary)
print_summary_table("Validation Set Scores (Baseline Reference)", val_summary)
print("=" * 72)


GMM BASELINE ANOMALY SCORE DIAGNOSTICS (-score_samples)

--- Training Set Scores (Diagnostics Only) ---
Cohort         |   Count |      Mean |       Std |      Min |   Median |      P95 |       Max
---------------------------------------------------------------------------------------------
Overall        | 170,888 |     43.26 |    111.69 |    27.90 |    35.05 |    55.79 |  10227.38
Legitimate     | 170,528 |     40.30 |     61.17 |    27.90 |    35.03 |    55.30 |  10227.38
Fraud          |     360 |   1441.40 |   1479.89 |    31.50 |   722.86 |  3800.68 |   6210.06

--- Validation Set Scores (Baseline Reference) ---
Cohort         |   Count |      Mean |       Std |      Min |   Median |      P95 |       Max
---------------------------------------------------------------------------------------------
Overall        |  56,957 |     45.30 |     72.86 |    29.15 |    38.07 |    61.01 |   5217.94
Legitimate     |  56,900 |     45.04 |     72.08 |    29.15 |    38.07 |    60.72 |   5217.9

## 11. Step 2 Summary & Baseline Status

- **Model:** One-component GMM with full covariance, reg_covar=$10^{-6}$, n_init=5, seed=42.
- **Fitting:** Fitted strictly on 170,528 legitimate training rows (`y_train == 0`).
- **Convergence:** Confirmed converged in 2 EM iterations (lower bound = -40.3046).
- **Scoring:** Negative log-likelihood computed for train (`170,888`) and validation (`56,957`).
- **Row IDs:** Preserved 1:1 with split manifest without positional coupling.
- **Reproducibility:** Confirmed identical across re-runs and serialization reload (`diff < 1e-10`).
- **Status:** **PASS** — Step 2 GMM Baseline implemented, verified, and saved reproducibly.
